# Machine Learning Foundations Lab\n\nCompanion to the first 50 questions in `knowledge-bank/foundations/ml-foundations-001-050.md`. Designed for a MacBook M1 with 8 GB RAM and for Google Colab. It uses a small synthetic dataset and scikit-learn; no GPU or model downloads are required.\n\n## Learning objectives\n- Build a reproducible train/test workflow.\n- Fit a preprocessing pipeline for numeric and categorical columns.\n- Compare a baseline and logistic regression.\n- Inspect precision, recall, and a confusion matrix.\n- Understand why all preprocessing must be fitted on training data only.\n

In [ ]:
import numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nimport matplotlib.pyplot as plt\n\nprint('NumPy:', np.__version__)\nprint('pandas:', pd.__version__)\nprint('scikit-learn:', sklearn.__version__)\n

## 1. Create a small, reproducible dataset\nWe simulate customer churn. This is synthetic teaching data, not evidence about real customers.

In [ ]:
rng = np.random.default_rng(42)\nn = 1200\ntenure = rng.integers(1, 73, size=n)\nmonthly_spend = rng.normal(65, 18, size=n).clip(10, 150)\nsupport_tickets = rng.poisson(1.5, size=n)\nplan = rng.choice(['basic', 'standard', 'premium'], size=n, p=[0.45, 0.4, 0.15])\nlogit = -1.2 - 0.035 * tenure + 0.32 * support_tickets + 0.012 * (monthly_spend - 60) + (plan == 'basic') * 0.45\nprobability = 1 / (1 + np.exp(-logit))\nchurn = rng.binomial(1, probability)\n\ndf = pd.DataFrame({\n    'tenure_months': tenure,\n    'monthly_spend': monthly_spend.round(2),\n    'support_tickets': support_tickets,\n    'plan': plan,\n    'churn': churn,\n})\n# Introduce a few missing values to demonstrate safe imputation.\nmissing_rows = rng.choice(df.index, size=35, replace=False)\ndf.loc[missing_rows, 'monthly_spend'] = np.nan\ndf.head()\n

In [ ]:
df['churn'].value_counts(normalize=True).rename('share').to_frame()\n

## 2. Split before fitting preprocessing\nThe test set is held aside. Imputers, encoders, and scalers are fitted only inside the training pipeline.

In [ ]:
X = df.drop(columns='churn')\ny = df['churn']\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)\n\nnumeric_features = ['tenure_months', 'monthly_spend', 'support_tickets']\ncategorical_features = ['plan']\n\nnumeric_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler()),\n])\ncategorical_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='most_frequent')),\n    ('onehot', OneHotEncoder(handle_unknown='ignore')),\n])\npreprocessor = ColumnTransformer([\n    ('numeric', numeric_pipeline, numeric_features),\n    ('categorical', categorical_pipeline, categorical_features),\n])\nprint(f'Train rows: {len(X_train):,}; test rows: {len(X_test):,}')\n

## 3. Compare a naive baseline with logistic regression\nThe baseline predicts the most frequent training class. Accuracy alone can be misleading, so we inspect class-specific precision and recall too.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')\nbaseline.fit(X_train, y_train)\nbaseline_predictions = baseline.predict(X_test)\nprint('Baseline report')\nprint(classification_report(y_test, baseline_predictions, zero_division=0))\n\nmodel = Pipeline([\n    ('preprocess', preprocessor),\n    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),\n])\nmodel.fit(X_train, y_train)\npredictions = model.predict(X_test)\nprobabilities = model.predict_proba(X_test)[:, 1]\nprint('Logistic regression report')\nprint(classification_report(y_test, predictions, zero_division=0))\nprint('ROC-AUC:', round(roc_auc_score(y_test, probabilities), 3))\n

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=['stays', 'churns'])\nplt.title('Logistic regression — held-out test set')\nplt.tight_layout()\nplt.show()\n

## 4. Try a different decision threshold\nThe default threshold is 0.5. A business should select a threshold using validation data and the cost of each error—not tune repeatedly on the final test set. This cell is a demonstration of the mechanics only.\n

In [ ]:
threshold = 0.35\nthreshold_predictions = (probabilities >= threshold).astype(int)\nprint(f'Threshold: {threshold:.2f}')\nprint(classification_report(y_test, threshold_predictions, zero_division=0))\n

## 5. Exercises\n1. Change `class_weight='balanced'` to `None`. Compare precision and recall.\n2. Remove the scaler from the numeric pipeline. Explain why it may affect logistic regression.\n3. Create a chronological split instead of a random split. What assumptions would that change?\n4. Explain why the threshold cell should not be used repeatedly to choose a threshold on the test set.\n5. Write down one feature that would cause target leakage if it were recorded after churn.\n\n## Key takeaways\n- Split first; fit preprocessing only on training data.\n- Keep preprocessing and the estimator in one pipeline.\n- Compare against a baseline and inspect more than accuracy.\n- Select thresholds and hyperparameters using validation procedures.\n- This synthetic exercise demonstrates mechanics; real deployment requires representative data, privacy review, subgroup evaluation, and monitoring.\n